# 03 — Validação do Hive

Notebook destinado à validação da camada de Data Warehouse da pipeline de e-commerce. São verificadas as bases, dimensões, fatos, tabelas externas, partições e views analíticas definidas no Hive.

**Camadas analisadas:** `ecommerce`, `ecommerce_staging` e `ecommerce_analytics`.


## 1. Configuração

Define os parâmetros utilizados para executar consultas no Hive.


In [ ]:
import os
import subprocess
from pathlib import Path
import pandas as pd

HIVE_BIN = os.getenv("HIVE_BIN", "hive")
HIVE_DATABASE = os.getenv("HIVE_DATABASE", "ecommerce")

DATABASES = ["ecommerce", "ecommerce_staging", "ecommerce_analytics"]

print(f"Executável Hive: {HIVE_BIN}")
print(f"Banco principal: {HIVE_DATABASE}")


## 2. Validação dos arquivos SQL e configuração Hive

Verifica se os arquivos responsáveis pela definição e manutenção do Data Warehouse estão presentes no projeto.


In [ ]:
project_root = Path.cwd()

files_to_check = [
    project_root / "configs/hive/hive-site.xml",
    project_root / "configs/hive/hive-env.sh",
    project_root / "sql/hive/ddl/01_create_database.sql",
    project_root / "sql/hive/ddl/02_create_dimensions.sql",
    project_root / "sql/hive/ddl/03_create_facts.sql",
    project_root / "sql/hive/ddl/04_create_external_tables.sql",
    project_root / "sql/hive/ddl/05_create_views.sql",
    project_root / "sql/hive/dml/load_daily_partitions.sql",
    project_root / "sql/hive/dml/refresh_views.sql",
    project_root / "sql/hive/dml/backfill.sql",
]

for path in files_to_check:
    print(f"{path}: {'OK' if path.exists() else 'NÃO ENCONTRADO'}")


## 3. Função para executar consultas Hive

In [ ]:
def hive_query(sql):
    result = subprocess.run(
        [HIVE_BIN, "-e", sql],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(
            result.stderr.strip() or "Falha na execução da consulta Hive"
        )
    return result.stdout.strip()


hive_available = True

try:
    print(hive_query("SHOW DATABASES;"))
except Exception as exc:
    hive_available = False
    print(f"Hive indisponível no momento: {exc}")


## 4. Validação dos bancos de dados

Confirma a existência das três camadas lógicas do Data Warehouse.


In [ ]:
if hive_available:
    databases_output = hive_query("SHOW DATABASES;")
    available = {
        line.strip()
        for line in databases_output.splitlines()
        if line.strip()
    }

    for database in DATABASES:
        print(
            f"{database}: "
            f"{'OK' if database in available else 'NÃO ENCONTRADO'}"
        )
else:
    print("Validação remota ignorada porque o Hive não está disponível.")


## 5. Dimensões do Data Warehouse

As dimensões utilizadas pela modelagem analítica são clientes, produtos, categorias e calendário.


In [ ]:
dimension_tables = [
    "dim_customer",
    "dim_product",
    "dim_category",
    "dim_date",
]

if hive_available:
    output = hive_query("USE ecommerce; SHOW TABLES;")
    tables = {line.strip() for line in output.splitlines() if line.strip()}

    for table in dimension_tables:
        print(f"{table}: {'OK' if table in tables else 'NÃO ENCONTRADA'}")
else:
    print("Dimensões serão verificadas com o Hive ativo.")


## 6. Tabelas fato

Verifica a estrutura das tabelas de vendas, cliques, carrinhos e entregas.


In [ ]:
fact_tables = [
    "fact_sales",
    "fact_clicks",
    "fact_carts",
    "fact_deliveries",
]

if hive_available:
    for table in fact_tables:
        print(f"\n===== {table} =====")
        try:
            print(hive_query(f"USE ecommerce; DESCRIBE {table};"))
        except Exception as exc:
            print(f"Não foi possível descrever {table}: {exc}")
else:
    print("Tabelas fato serão verificadas com o Hive ativo.")


## 7. Tabelas externas de staging

Verifica as tabelas externas que representam dados provenientes das camadas HDFS.


In [ ]:
external_tables = [
    "external_raw_events",
    "external_clean_events",
    "external_curated_events",
    "external_streaming_metrics",
    "external_streaming_alerts",
]

if hive_available:
    output = hive_query("USE ecommerce_staging; SHOW TABLES;")
    tables = {line.strip() for line in output.splitlines() if line.strip()}

    for table in external_tables:
        print(f"{table}: {'OK' if table in tables else 'NÃO ENCONTRADA'}")
else:
    print("Tabelas externas serão verificadas com o Hive ativo.")


## 8. Particionamento temporal

As tabelas fato e externas utilizam `event_date` como partição para organizar o histórico temporal e reduzir o volume processado nas consultas.


In [ ]:
partitioned_tables = fact_tables + external_tables

if hive_available:
    for table in partitioned_tables:
        database = (
            "ecommerce"
            if table.startswith("fact_")
            else "ecommerce_staging"
        )

        print(f"\n===== {database}.{table} =====")

        try:
            result = hive_query(
                f"USE {database}; SHOW PARTITIONS {table};"
            )
            print(result or "Nenhuma partição registrada.")
        except Exception as exc:
            print(f"Não foi possível consultar partições: {exc}")
else:
    print("Partições serão consultadas com o Hive ativo.")


## 9. Views analíticas

As views da camada `ecommerce_analytics` consolidam indicadores para consumo analítico.


In [ ]:
views = [
    "vw_sales_daily",
    "vw_sales_by_product",
    "vw_customer_sales",
    "vw_customer_behavior",
    "vw_cart_behavior",
    "vw_delivery_performance",
    "vw_sales_delivery",
]

if hive_available:
    output = hive_query("USE ecommerce_analytics; SHOW VIEWS;")
    available_views = {
        line.strip()
        for line in output.splitlines()
        if line.strip()
    }

    for view in views:
        print(
            f"{view}: "
            f"{'OK' if view in available_views else 'NÃO ENCONTRADA'}"
        )
else:
    print("Views serão verificadas com o Hive ativo.")


## 10. Validação das principais consultas analíticas

In [ ]:
metric_queries = {
    "vendas diárias":
        "SELECT * FROM ecommerce_analytics.vw_sales_daily LIMIT 10",
    "vendas por produto":
        "SELECT * FROM ecommerce_analytics.vw_sales_by_product LIMIT 10",
    "vendas por cliente":
        "SELECT * FROM ecommerce_analytics.vw_customer_sales LIMIT 10",
    "comportamento":
        "SELECT * FROM ecommerce_analytics.vw_customer_behavior LIMIT 10",
    "carrinhos":
        "SELECT * FROM ecommerce_analytics.vw_cart_behavior LIMIT 10",
    "logística":
        "SELECT * FROM ecommerce_analytics.vw_delivery_performance LIMIT 10",
}

if hive_available:
    for name, query in metric_queries.items():
        print(f"\n===== {name} =====")
        try:
            result = hive_query(query)
            print(result or "Sem registros.")
        except Exception as exc:
            print(f"Consulta indisponível: {exc}")
else:
    print("As métricas serão consultadas com o Hive ativo.")


## 11. Contagem dos registros das tabelas fato

A contagem fornece uma visão rápida do volume consolidado em cada fato.


In [ ]:
if hive_available:
    counts = []

    for table in fact_tables:
        try:
            output = hive_query(
                f"SELECT COUNT(*) FROM ecommerce.{table};"
            )
            value = int(output.splitlines()[-1])
            counts.append({
                "table": table,
                "total_records": value,
            })
        except Exception as exc:
            counts.append({
                "table": table,
                "total_records": None,
                "error": str(exc),
            })

    display(pd.DataFrame(counts))
else:
    print("Contagens serão executadas com o Hive ativo.")


## 12. Validação do modelo analítico

A camada `ecommerce` concentra dimensões e fatos. A camada `ecommerce_staging` mantém tabelas externas associadas ao HDFS. A camada `ecommerce_analytics` disponibiliza as views para análise.


In [ ]:
expected_model = {
    "dimensions": dimension_tables,
    "facts": fact_tables,
    "external_tables": external_tables,
    "views": views,
}

for layer, objects in expected_model.items():
    print(f"{layer}: {len(objects)} objetos definidos")
    for obj in objects:
        print(f"  - {obj}")


## 13. Conclusão

A validação deste notebook demonstra a organização da camada Hive em dimensões, fatos, tabelas externas e views analíticas. O particionamento por `event_date` organiza o histórico temporal, enquanto as views consolidam indicadores de vendas, comportamento e logística para consumo analítico.
